# Metastasis free survival prediction using transcriptomics and proteomics data from the `APOLLO-LUAD` study

## Approach 
- Based on multimodal cancer modeling, described here https://arxiv.org/abs/2505.07683
- Fit CoxPH model on zero-shot embeddings extracted from survival protein and survival RNA expression from `APOLLO-LUAD`
- `APOLLO-LUAD` paper, `Soltis et al 2022`  https://gdc.cancer.gov/about-data/publications/APOLLO-LUAD-2022

## Analysis scheme

### Survival RNA expression
- get RNA expression data for each patient from GDC portal
- embed with bulkRNABERT model n=256 dims, only considering expression for survival RNAs (n=155)
- embeddings accessible via APIs, pull from API and join with usable clinical data ( MFS survival data )
- fit coxPH model: X => RNA expressions, y -> Dead/Alive, survival time in months for metastasis free survival (MFS)
- split data into train and test (0.2)
- scale training data and reduce using PCA (`X_train_red_rna`)
- fit cox model `cox_rna` on reduced training data
- use `cox_rna` to predict risk scores on `X_test_red_rna` (`y_test_pred_rna`)
- calculate C-index between `y_test_rna` and `y_test_pred_rna`

### Survival Protein
- repeat the same analysis as RNA, except with survival protein data (n=560) and PCA embeddings

###  Survival RNA + Survival Protein
- combine risk scores from the two modalities, fit coxPH model and recalculate C-index

### All RNAs
- same analysis as survival RNA expression, but using all RNAs (n ~ 19K)

### All Proteins
- same analysis as survival proteins, but using all proteins (n ~ 7.6K)

### All RNAs + All Proteins
- combine risk scores from the two modalities, fit coxPH model and recalculate C-index

### C-index comparisons

#### Compare
- C-index from survival RNA-Seq embeddings from bulkRNABERT
- C-index from survival protein embeddings from PCA
- C-index from survival RNA-Seq + survival proteins (proteomics + transcriptomics)
- C-index from all RNA embeddings
- C-index from all protein embeddings
- C-index from all RNA embeddings + all protein embeddings

### Combined plot
- calculate a median risk score and partition the test set into high risk (>median) and low risk (< median)
- plot survival curves for the two groups for various combinations above

### Path to precomputed embeddings and supplementary files

In [ ]:
cred_root = '/home/jovyan/pd/'
data_root = './data'

### Import libraries

In [ ]:
import pandas as pd
import os
import numpy as np
import requests
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
from tqdm import tqdm
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.util import Surv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import concordance_index_censored
from gen3.auth import Gen3Auth


In [ ]:
tqdm.pandas()

## Survival RNA analysis

### Read supplementary table S2C with MFS data

make the links

https://www.cell.com/cell-reports-medicine/fulltext/S2666-3791(22)00378-0?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS2666379122003780%3Fshowall%3Dtrue

https://www.cell.com/cms/10.1016/j.xcrm.2022.100819/attachment/700c2b3c-4bcf-45c2-b413-ead3903e4165/mmc2.xls

In [ ]:
table_s2c = pd.read_excel(os.path.join(data_root, '1-s2.0-S2666379122003780-mmc3.xlsx'), sheet_name='TableS2C.APOLLO.MFS')

### Retrieve survival RNA embeddings using the MC2DP embedding service

In [ ]:
auth = Gen3Auth(refresh_file=os.path.join(cred_root, 'mc2dp-production-credentials.json'))

In [ ]:
EMBEDDING_API = "https://mc2dp.data-commons.org/ai/vectorstore/collections"

col_meta = requests.get(f"{EMBEDDING_API}/rna-seq-survival", params={"counts": True}, auth=auth).json()
n = col_meta["available_embeddings_count"]

r = requests.get(f"{EMBEDDING_API}/rna-seq-survival/embeddings", params={"page": 1}, auth=auth)
r.raise_for_status()

expr_embs = dict()
for x in tqdm(r.json()["embeddings"]):
    #print(x)
    fid = x["info"]["metadata"]["file_id"]
    expr_embs[fid] = np.asarray(x["vector"])

print(len(expr_embs.keys()))

### Read clinical data for MFS from supplement
- create survival MFS time in months

In [ ]:
table_s1 = pd.read_excel(os.path.join(data_root, '1-s2.0-S2666379122003780-mmc2.xls'), sheet_name='Table S1A Sample Information')

In [ ]:
average_days_per_month = 30.44
table_s1['surv_mfs_time_months'] = table_s1['surv_mfs_time'] / average_days_per_month

In [ ]:
table_s1.head(n=2)

### Join survival RNA embeddings with clinical data 
- map `file_id` from embeddings to `submitter_id`

In [ ]:
def map_file_id_to_submitter_id(file_name, suffix):
    file_name = file_name + suffix
    api_endpt = 'https://api.gdc.cancer.gov/files'
    fields = [
        "cases.submitter_id"
    ]
    fields = ",".join(fields)

    filters = {
        "op": "and",
          "content": [
              {
                "op": "in",
                "content": {"field": "file_name", "value": file_name},
              }
          ]
    }
    params = {"filters": json.dumps(filters), "fields": fields, "size": 10}
    try:
      response = requests.get(api_endpt, params=params)
      response_json = json.loads(response.content)
      submitter_id = response_json['data']['hits'][0]['cases'][0]['submitter_id']
    except Exception as e:
       print(f'unable to execute request, exception {str(e)}')
    return submitter_id

In [ ]:
expr_df = pd.DataFrame(list(embs.keys()), columns=['file_id'])

In [ ]:
expr_df.head()

In [ ]:
expr_df[expr_df['submitter_id'].isnull()]

In [ ]:
expr_df_mfs = pd.merge(
    table_s1[['caseId', 'surv_mfs_usable']],
    expr_df,
    left_on=['caseId'],
    right_on=['submitter_id'],
    how='inner'
)

In [ ]:
expr_df_mfs.head()

In [ ]:
expr_df_mfs.shape

### Extract embeddings for each case and create matrix for training using usable clinical data

In [ ]:
def extract_case_emb(embs, file_ids: list[str]):
    X = []
    for fid in tqdm(file_ids):
        # if there are more than one embeddings stack them
        dims = embs[fid].ndim
        if dims > 1:
            embs = np.stack([embs[fid][i] for i in range(dims)], axis=0)
            emb = np.mean(embs, axis=0)
        else:
            emb = embs[fid]
        X.append(emb)
    return np.stack(X, axis=0)

In [ ]:
mfs_usable_file_ids = list(expr_df_mfs[expr_df_mfs['surv_mfs_usable'] == True]['file_id'])

In [ ]:
len(mfs_usable_file_ids)

In [ ]:
expr_X = extract_case_emb(embs, mfs_usable_file_ids)

In [ ]:
expr_X.shape

In [ ]:
expr_X[:3, :3]

### Ensure order of cases in clinical data matches expression matrix

In [ ]:
table_s1_usable = table_s1[table_s1['surv_mfs_usable']]

In [ ]:
table_s1_usable.shape

In [ ]:
table_s1_usable = table_s1_usable[['caseId', 'surv_mfs_stat', 'surv_mfs_time', 'surv_mfs_time_months', 'surv_mfs_usable']]

In [ ]:
mfs_usable_submitter_ids = list(expr_df_mfs[expr_df_mfs['surv_mfs_usable'] == True]['submitter_id'])
mfs_usable_submitter_ids == table_s1_usable['caseId'].to_list()

### Fit CoxPH model on survival RNA embeddings

In [ ]:
y = Surv.from_arrays(event=table_s1_usable['surv_mfs_stat'].astype(bool),
                     time=table_s1_usable['surv_mfs_time_months']
                     )
X = expr_X

In [ ]:
# train/test split
X_train_rna, X_test_rna, y_train_rna, y_test_rna = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()
X_train_scaled_rna = scaler.fit_transform(X_train_rna)
X_test_scaled_rna = scaler.transform(X_test_rna)

In [ ]:
X_train_scaled_rna.shape

In [ ]:
X_test_scaled_rna.shape

In [ ]:
pca_components = 66
pca = PCA(n_components=pca_components, random_state=42, svd_solver='full')
X_train_red_rna = pca.fit_transform(X_train_scaled_rna)
X_test_red_rna = pca.transform(X_test_scaled_rna)

In [ ]:
cox_rna = CoxPHSurvivalAnalysis(alpha=0.1).fit(X_train_red_rna, y_train_rna)

In [ ]:
cox_rna

In [ ]:
# predict risk scores
y_train_pred_rna = cox_rna.predict(X_train_red_rna)
y_test_pred_rna = cox_rna.predict(X_test_red_rna)

In [ ]:
# extract survival time and bool for death
y_test_events = np.array([bool(data[0]) for data in y_test_rna])
y_test_time = np.array([data[1] for data in y_test_rna])

### Calculate C-index for survival RNAs

In [ ]:
c_index = concordance_index_censored(
    event_indicator=y_test_events,
    event_time=y_test_time,
    estimate=y_test_pred_rna,
)[0]


In [ ]:
c_index

## Survival Protein analysis

### Read data for survival proteins and create dataframe

In [ ]:
protein_expr = pd.read_csv(os.path.join(data_root, 'APOLLO1_Level3_Gprot_v061418.n87.csv'), index_col=0)

In [ ]:
protein_expr.shape

In [ ]:
protein_expr.head(n=2)

In [ ]:
survival_proteins = table_s2c[table_s2c['Protein_signature'] == 1]

In [ ]:
survival_proteins.shape

In [ ]:
survival_proteins.head(n=2)

In [ ]:
survival_proteins['accession'] = survival_proteins['Gene'].apply(
    lambda x: protein_expr[protein_expr['Gene'] == x]['Accession'].iloc[0]
)

In [ ]:
survival_protein_names = survival_proteins['accession'].to_list()

In [ ]:
len(set(survival_protein_names))

### Rearrange dataframe for PCA

In [ ]:
protein_expr_t = protein_expr.T.reset_index().iloc[3:,:]

In [ ]:
new_colnames = ['protein_sample_id'] + protein_expr['Accession'].to_list()
protein_expr_t.columns = new_colnames
protein_expr_t.head(n=3)

### Join protein expression data with clinical data

In [ ]:
# align case ids in protein_expr data with Table S1A
protein_expr_t['caseId'] = protein_expr_t['protein_sample_id'].apply(
    lambda x: '-'.join(x.split('.')[:2])
)

In [ ]:
protein_expr_t.head(n=3)

In [ ]:
protein_expr_with_clinical = pd.merge(
    table_s1_usable,
    protein_expr_t,
    on=['caseId'],
    how='inner'
)

In [ ]:
protein_expr_with_clinical.head(n=3)

In [ ]:
protein_expr_with_clinical.shape

In [ ]:
# ensure order of samples matches in clinical data and protein expression data
protein_expr_with_clinical['caseId'].to_list() == table_s1_usable['caseId'].to_list()

### Format data for PCA

In [ ]:
drop_columns = list(table_s1_usable.columns) + ['protein_sample_id']
protein_expression_data = protein_expr_with_clinical.drop(columns=drop_columns)

In [ ]:
protein_expression_data.head(n=3)

In [ ]:
survival_protein_expression = protein_expression_data[survival_protein_names]

In [ ]:
survival_protein_expression.head(n=3)

In [ ]:
survival_protein_expression.shape

### Fit CoxPH model on survival protein embeddings

In [ ]:
# convert survival data to structured array
y_surv_protein = Surv.from_arrays(event=table_s1_usable['surv_mfs_stat'].astype(bool),
                     time=table_s1_usable['surv_mfs_time_months']
                     )
X_surv_protein = survival_protein_expression

In [ ]:
# train/test split
X_train_protein, X_test_protein, y_train_protein, y_test_protein = train_test_split(
    X_surv_protein, y_surv_protein, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()
X_train_scaled_protein = scaler.fit_transform(X_train_protein)
X_test_scaled_protein = scaler.transform(X_test_protein)

In [ ]:
X_train_scaled_protein.shape

In [ ]:
X_test_scaled_protein.shape

### Reduce with PCA and fit model

In [ ]:
pca_components = 66
pca_p = PCA(n_components=pca_components, random_state=42, svd_solver='full')
X_train_red_protein = pca_p.fit_transform(X_train_scaled_protein)
X_test_red_protein = pca_p.transform(X_test_scaled_protein)

In [ ]:
cox_protein = CoxPHSurvivalAnalysis(alpha=0.1).fit(X_train_red_protein, y_train_protein)

In [ ]:
cox_protein

In [ ]:
# predict risk scores
y_train_pred_protein = cox_protein.predict(X_train_red_protein)
y_test_pred_protein = cox_protein.predict(X_test_red_protein)

In [ ]:
y_test_events_p = np.array([bool(data[0]) for data in y_test_protein])
y_test_time_p = np.array([data[1] for data in y_test_protein])

### Calculate C-index for survival proteins

In [ ]:
c_index_p = concordance_index_censored(
    event_indicator=y_test_events_p,
    event_time=y_test_time_p,
    estimate=y_test_pred_protein,
)[0]

In [ ]:
c_index_p

## Combined analysis

### Merge RNA and protein risk scores and recalculate C-index

In [ ]:
combos = [['rna', 'protein']]
for combo in combos:
    mult_X_train = []
    mult_X_test = []
    for modality in combo:
        train_data = '_'.join(['y_train_pred', modality])
        x_train = eval(train_data)[:, np.newaxis]
        test_data = '_'.join(['y_test_pred', modality])
        x_test = eval(test_data)[:, np.newaxis]
        scaler = StandardScaler()
        x_train = scaler.fit_transform(x_train)
        x_test = scaler.transform(x_test)
        mult_X_train.append(x_train)
        mult_X_test.append(x_test)

    mult_X_train = np.concatenate(mult_X_train, axis=1)
    mult_X_test = np.concatenate(mult_X_test, axis=1)

In [ ]:
mult_X_train[:5, :]

In [ ]:
mult_X_test[:5, :]

In [ ]:
cox_combined = CoxPHSurvivalAnalysis(alpha=0.1).fit(mult_X_train, y_train_rna)

In [ ]:
cox_combined

In [ ]:
# generate predictions
y_train_pred_combined = cox_combined.predict(mult_X_train)
y_test_pred_combined = cox_combined.predict(mult_X_test)

### Calculate combined C-index

In [ ]:
c_index_combined = concordance_index_censored(
    event_indicator=y_test_events,
    event_time=y_test_time,
    estimate=y_test_pred_combined,
)[0]

In [ ]:
c_index_combined

## All RNA analysis
- Utilizing the Embedding service with precomputed embeddings for all genes (n=19,062)

In [ ]:
col_meta = requests.get(f"{EMBEDDING_API}/rna-seq", params={"counts": True}, auth=auth).json()
n = col_meta["available_embeddings_count"]

r = requests.get(f"{EMBEDDING_API}/rna-seq/embeddings", params={"page": 1}, auth=auth)
r.raise_for_status()

all_rna_embs = dict()
for x in tqdm(r.json()["embeddings"]):
    fid = x["info"]["metadata"]["file_id"]
    all_rna_embs[fid] = np.asarray(x["vector"])

print(len(all_rna_embs.keys()))

In [ ]:
all_rna_expr_df_mfs = pd.merge(
    table_s1[['caseId', 'surv_mfs_usable']],
    expr_df,
    left_on=['caseId'],
    right_on=['submitter_id'],
    how='inner'
)
mfs_usable_case_ids = list(all_rna_expr_df_mfs[all_rna_expr_df_mfs['surv_mfs_usable'] == True]['file_id'])
all_rna_expr_X = extract_case_emb(all_rna_embs, mfs_usable_case_ids)

In [ ]:
# ensure correct order of cases in all_rna expr and clinical data
mfs_usable_submitter_ids = list(all_rna_expr_df_mfs[all_rna_expr_df_mfs['surv_mfs_usable'] == True]['submitter_id'])
mfs_usable_submitter_ids == table_s1_usable['caseId'].to_list()

In [ ]:
# train/test split
X_train_rna_all, X_test_rna_all, y_train_rna_all, y_test_rna_all = train_test_split(
    X_rna_all, y_rna_all, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_scaled_rna_all = scaler.fit_transform(X_train_rna_all)
X_test_scaled_rna_all = scaler.transform(X_test_rna_all)
pca_components = 66
pca_all = PCA(n_components=pca_components, random_state=42, svd_solver='full')
X_train_red_rna_all = pca_all.fit_transform(X_train_scaled_rna_all)  # was pca
X_test_red_rna_all = pca_all.transform(X_test_scaled_rna_all)        # was pca
# fit model
cox_rna_all = CoxPHSurvivalAnalysis(alpha=0.1).fit(X_train_red_rna_all, y_train_rna_all)
# predict risk scores
y_train_pred_rna_all = cox_rna_all.predict(X_train_red_rna_all)
y_test_pred_rna_all = cox_rna_all.predict(X_test_red_rna_all)
y_test_events_all = np.array([bool(data[0]) for data in y_test_rna_all])
y_test_time_all = np.array([data[1] for data in y_test_rna_all])
c_index_rna_all = concordance_index_censored(
    event_indicator=y_test_events_all,
    event_time=y_test_time_all,
    estimate=y_test_pred_rna_all,
)[0]
c_index_rna_all

## All Protein analysis

In [ ]:
y_surv_protein_all = Surv.from_arrays(event=table_s1_usable['surv_mfs_stat'].astype(bool),
                     time=table_s1_usable['surv_mfs_time_months']
                     )
X_surv_protein_all = protein_expression_data

In [ ]:
# train/test split
X_train_protein_all, X_test_protein_all, y_train_protein_all, y_test_protein_all = train_test_split(
    X_surv_protein_all, y_surv_protein_all, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_scaled_protein_all = scaler.fit_transform(X_train_protein_all)
X_test_scaled_protein_all = scaler.transform(X_test_protein_all)
# reduce with pca
pca_components = 66
pca_p_all = PCA(n_components=pca_components, random_state=42, svd_solver='full')
X_train_red_protein_all = pca_p.fit_transform(X_train_scaled_protein_all)
X_test_red_protein_all = pca_p.transform(X_test_scaled_protein_all)
# fit model
cox_protein_all = CoxPHSurvivalAnalysis(alpha=0.1).fit(X_train_red_protein_all, y_train_protein_all)
# predict risk scores
y_train_pred_protein_all = cox_protein_all.predict(X_train_red_protein_all)
y_test_pred_protein_all = cox_protein_all.predict(X_test_red_protein_all)
y_test_events_p_all = np.array([bool(data[0]) for data in y_test_protein_all])
y_test_time_p_all = np.array([data[1] for data in y_test_protein_all])
# calculate c_index
c_index_p_all = concordance_index_censored(
    event_indicator=y_test_events_p_all,
    event_time=y_test_time_p_all,
    estimate=y_test_pred_protein_all,
)[0]
c_index_p_all

## Combine all RNA and all protein risk scores
- recalculate c-index

In [ ]:
combos = [['rna_all', 'protein_all']]
for combo in combos:
    mult_X_train_all = []
    mult_X_test_all = []
    for modality in combo:
        train_data = '_'.join(['y_train_pred', modality])
        x_train = eval(train_data)[:, np.newaxis]
        test_data = '_'.join(['y_test_pred', modality])
        x_test = eval(test_data)[:, np.newaxis]
        scaler = StandardScaler()
        x_train = scaler.fit_transform(x_train)
        x_test = scaler.transform(x_test)
        mult_X_train_all.append(x_train)
        mult_X_test_all.append(x_test)

    mult_X_train_all = np.concatenate(mult_X_train_all, axis=1)
    mult_X_test_all = np.concatenate(mult_X_test_all, axis=1)

In [ ]:
cox_combined_all = CoxPHSurvivalAnalysis(alpha=0.1).fit(mult_X_train_all, y_train_rna_all)
# generate predictions
y_train_pred_combined_all = cox_combined_all.predict(mult_X_train_all)
y_test_pred_combined_all = cox_combined_all.predict(mult_X_test_all)
c_index_combined_all = concordance_index_censored(
    event_indicator=y_test_events,
    event_time=y_test_time,
    estimate=y_test_pred_combined_all,
)[0]
c_index_combined_all

## Best prediction is achieved using survival RNAs + survival proteins

### C-index summary

In [ ]:
print('Unimodal results')
print('------------------')
print('survival RNAs: 0.69')
print('all RNAs: 0.53')

print('survival proteins: 0.77')
print('all proteins: 0.75')
print('\n')
print('Multimodal results')
print('------------------')
print('survival RNAs + survival proteins: 0.77')
print('all RNAs + all proteins: 0.65')

## Plot
- stratify cases by median risk scores calculated from the different analysis on protein and RNA data
- plot low risk and high risk groups based on median

### calculate risk scores

In [ ]:
median_risk_score_rna = np.median(y_test_pred_rna)
median_risk_score_protein = np.median(y_test_pred_protein)
median_risk_score_combined = np.median(y_test_pred_combined)
median_risk_score_rna_all = np.median(y_test_pred_rna_all)
median_risk_score_protein_all = np.median(y_test_pred_protein_all)
median_risk_score_combined_all = np.median(y_test_pred_combined_all)

mask_low_rna = y_test_pred_rna <= median_risk_score_rna
mask_high_rna = y_test_pred_rna > median_risk_score_rna

mask_low_p = y_test_pred_protein <= median_risk_score_protein
mask_high_p = y_test_pred_protein > median_risk_score_protein

mask_low_comb = y_test_pred_combined <= median_risk_score_combined
mask_high_comb = y_test_pred_combined > median_risk_score_combined

mask_low_rna_all = y_test_pred_rna_all <= median_risk_score_rna_all
mask_high_rna_all = y_test_pred_rna_all > median_risk_score_rna_all

mask_low_p_all = y_test_pred_protein_all <= median_risk_score_protein_all
mask_high_p_all = y_test_pred_protein_all > median_risk_score_protein_all

mask_low_comb_all = y_test_pred_combined_all <= median_risk_score_combined_all
mask_high_comb_all = y_test_pred_combined_all > median_risk_score_combined_all


### Create low-risk and high-risk groups

In [ ]:
# survival protein proteomics: Low-risk group
time_low_p, surv_low_p = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_low_p].astype(bool),
    y_test_time[mask_low_p]
)
# survival protein proteomics: High-risk group
time_high_p, surv_high_p = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_high_p].astype(bool),
    y_test_time[mask_high_p]
)

# all proteins : Low-risk group
time_low_p_all, surv_low_p_all = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_low_p_all].astype(bool),
    y_test_time[mask_low_p_all]
)
# all proteins: High-risk group
time_high_p_all, surv_high_p_all = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_high_p_all].astype(bool),
    y_test_time[mask_high_p_all]
)

# surv rna: Low-risk group
time_low_rna, surv_low_rna = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_low_rna].astype(bool),
    y_test_time[mask_low_rna]
)
# surv rna: High-risk group
time_high_rna, surv_high_rna = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_high_rna].astype(bool),
    y_test_time[mask_high_rna]
)

# all rnas: Low-risk group
time_low_rna_all, surv_low_rna_all = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_low_rna_all].astype(bool),
    y_test_time[mask_low_rna_all]
)
# all rnas: High-risk group
time_high_rna_all, surv_high_rna_all = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_high_rna_all].astype(bool),
    y_test_time[mask_high_rna_all]
)

# surv combined: Low-risk group
time_low_combined, surv_low_combined = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_low_comb].astype(bool),
    y_test_time[mask_low_comb]
)
# surv combined: High-risk group
time_high_combined, surv_high_combined = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_high_comb].astype(bool),
    y_test_time[mask_high_comb]
)

# combined all: Low-risk group
time_low_combined_all, surv_low_combined_all = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_low_comb_all].astype(bool),
    y_test_time[mask_low_comb_all]
)
# combined all: High-risk group
time_high_combined_all, surv_high_combined_all = kaplan_meier_estimator(
    y_test_events.astype(int)[mask_high_comb_all].astype(bool),
    y_test_time[mask_high_comb_all]
)

### Plots

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 6))

# low-risk/high-risk group survival proteins
axes[0, 0].step(time_low_p, surv_low_p, where="post", color='blue')
axes[0, 0].step(time_high_p, surv_high_p, where="post", color='orange')
axes[0 ,0].set_title('survival proteins')
axes[0, 0].legend(loc="upper right", title="--cindex=0.77", labels=[],fontsize=9, frameon=False)
axes[0, 0].grid(True)

# low-risk/high-risk group survival rnas
axes[0, 1].step(time_low_rna, surv_low_rna, where="post", color='blue')
axes[0, 1].step(time_high_rna, surv_high_rna, where="post", color='orange')
axes[0 ,1].set_title('survival RNAs')
axes[0, 1].legend(loc="upper right", title="--cindex=0.69", labels=[], fontsize=9, frameon=False)
axes[0, 1].grid(True)

# low-risk/high-risk group combined survival proteins + survival rnas
axes[0, 2].step(time_low_combined, surv_low_combined, where="post", color='blue')
axes[0, 2].step(time_high_combined, surv_high_combined, where="post", color='orange')
axes[0 ,2].set_title('survival proteins + survival RNAs')
axes[0, 2].legend(loc="upper right", title="--cindex=0.77", labels=[], fontsize=9, frameon=False)
axes[0, 2].grid(True)

# low-risk/high-risk group all proteins
axes[1, 0].step(time_low_p_all, surv_low_p_all, where="post", color='blue')
axes[1, 0].step(time_high_p_all, surv_high_p_all, where="post", color='orange')
axes[1 ,0].set_title('all proteins')
axes[1, 0].legend(loc="upper right", title="--cindex=0.75", labels=[], fontsize=9, frameon=False)
axes[1, 0].grid(True)

# low-risk/high-risk group all rnas
axes[1, 1].step(time_low_rna_all, surv_low_rna_all, where="post", color='blue', )
axes[1, 1].step(time_high_rna_all, surv_high_rna_all, where="post", color='orange')
axes[1 ,1].set_title('all RNAs')
axes[1, 1].legend(loc="upper right", title="--cindex=0.53", labels=[], fontsize=9, frameon=False)
axes[1, 1].grid(True)

# low-risk/high-risk group all proteins + all rnas
axes[1, 2].step(time_low_combined_all, surv_low_combined_all, where="post", color='blue')
axes[1, 2].step(time_high_combined_all, surv_high_combined_all, where="post", color='orange')
axes[1, 2].set_title('all proteins + all RNAs')
axes[1, 2].legend(loc="upper right", title="--cindex=0.65", labels=[], fontsize=9, frameon=False)
axes[1, 2].grid(True)

plt.xlabel("Time (months)")
plt.ylabel("Survival probability")

fig.suptitle(
    "Kaplan–Meier survival curves by predicted risk group",
)
plt.show()

## Conclusions
- PCA embeddings from just n=560 survival proteins produce equivalent or better risk score predictions than PCA embeddings from all proteins (n=7.6k)
- BulkRNABERT embeddings from just n=155 survival RNAs produce better risk score predictions than BulkRNABERT embeddings from all RNAs (n=19k)
- Survival protein and survival RNA embeddings better predict patient survival than all proteins or RNAs, consistent with inferences of Soltis et al. 2022
- Proteomics data is a powerful additional modality to include for multimodal survival analysis, best c-index is obtained using survival proteins, and combination of survival proteins + survival RNAs
- MFS endpoint is analyzed, same analysis can be applied to overall survival